In [ ]:
# SPDX-License-Identifier: GPL-3.0-or-later

# Workflow to perform full-field recrystallization simulations

## Contextualization

This notebook describes the workflow as it is described in the paper **Open-source simulation and research data management tools combined for studying recrystallization in industrial aluminium alloys**

The workflow here follows the figure 1 in the paper. 

<div class="alert alert-info">
The notebook describes the workflow and does not aim to recreate the simulations in the paper.
</div>

Therefore, the following should be considered:
1. The RVE considered here is smaller.
2. The deformation level applied here is smaller. Consequently, there are fewer remeshing steps. For running this example quickly even on modest hardware two deformation steps with 0.1 compressive strain each are taken into account. Between the two deformation steps, there is a remeshing step. For the sake of readability, the code blocks for running the simulations are shortened by using helper functions.
3. The source code to evaluate the nucleation criterion that is described in the paper is not included in the SCORE version as that code is proprietary.

In [ ]:
import os, sys
import subprocess

cwd_path = os.getcwd()
print(cwd_path)
# sys.path.append(f"{cwd_path}/scripts/")

## Synthetic microstructure generation (DREAM3D)

The microstructure parameters used to create the synthetic microstructure are described in Section 2.2.2 and 2.2.4 in the paper.

The average grain size (ESD) compares to the measured grain size (ECD). The average precipitate size (ESD) also compares to the measured precipitate size (ECD). Moreover, the volume fraction of the second phase correspond to the area fraction of the second phase measured.

Here the corresponding DREAM3D pipeline file is run to create the microstructure.

The pipeline is pre-designed to create a representative volume element (RVE) of a two phase microstructure that is discretized using 15 cells each along the rolling direction, the transverse direction, and the normal direction. The voxel size is 0.8 micrometer. In case of changing the microstructure generation settings please look at the documentation of [DREAM3D](https://www.dream3d.io/python_docs/index.html).

In [ ]:
import simplnx as nx
dream_3d_bin_path = "CHANGEME"

pipeline_file = f"{cwd_path}/example_files/Microstructure_instantiation_pipeline.json"

<div class="alert alert-warning">
DREAM3D-NX not supporting old pipeline files
</div>

In [ ]:
# os.chdir(dream_3d_bin_path)                                                                                             
cmd = f'./PipelineRunner -p {pipeline_file}'                                                                   
p = subprocess.Popen(cmd, shell=True)                                                                                    
while p.poll() == None:
    p.poll()                                                                                                        
if p.poll() == 0:                                                                                                       
   print(f'Synthetic microstructure created!!')

In [ ]:
# Path to your DREAM3D / simplnx pipeline JSON file
"""
data_structure = nx.DataStructure()
pipeline = nx.Pipeline().from_file(pipeline_file)
result = pipeline.execute(data_structure)
if result.result.invalid():
    print("Pipeline execution failed")
    for err in result.result.errors:
        print(err)
else:
    print("Pipeline executed successfully")
"""

## Crystal plasticity simulation (DAMASK)

### Import DREAM3D file and parameterize the DAMASK simulation

In [ ]:
# os.chdir(cwd_path + '/simulation/')

In [ ]:
from damask import GeomGrid
from damask import ConfigMaterial
import damask
simulation_path = f"{cwd_path}/simulation"

Create the RVE geometry file how to discretize the RVE into material points.

In [ ]:
geom = GeomGrid.load_DREAM3D(
    f"{simulation_path}/Microstructure_for_paper.dream3d"
)
    # feature_IDs="/DataContainers/SyntheticVolumeDataContainer/CellData"
# )

In [ ]:
geom.size = geom.size * 1E-06

In [ ]:
geom.save(f"{simulation_path}/Microstructure_for_paper.vti")
geom

Create the material file

In [ ]:
mat = ConfigMaterial.load_DREAM3D(
    f"{simulation_path}/Microstructure_for_paper.dream3d",
    base_group="/DataContainers/SyntheticVolumeDataContainer"
    # grain_data="/DataContainers/SyntheticVolumeDataContainer/Grain Data"
)

In [ ]:
mat.save(f"{simulation_path}/material.yaml")

The material file created at this step has only the grain orientation and the assigned phase for each material point. We define the material parameters for each of these phases.

In [ ]:
example_mat = ConfigMaterial.load(f"{cwd_path}/example_files/example_material.yaml")

In [ ]:
mat = ConfigMaterial.load(f"{simulation_path}/material.yaml")

In [ ]:
mat['phase']['Aluminum'] = example_mat['phase']['Aluminum']

In [ ]:
mat['phase']['Second_phase'] = example_mat['phase']['Second_phase']

In [ ]:
mat['homogenization']['direct'] = example_mat['homogenization']['SX']

In [ ]:
mat.save(f"{simulation_path}/material.yaml")

### Run the first deformation step (DAMASK)

In [ ]:
geom_file = f"{simulation_path}/Microstructure_for_paper"
load_file = f"{simulation_path}/tensionX"
mat_file = f"{simulation_path}/material"
max_processes = 1
# max_threads = int(os.cpu_count()/2 if os.cpu_count() >= 2 else 1)
print(f"Running with {max_processes} MPI processes")
# ! export OMP_NUM_THREADS=$max_threads
! echo $OMP_NUM_THREADS

In [ ]:
from jupyterlab_h5web import H5Web
H5Web(f"Microstructure_for_paper.dream3d")

In [ ]:
cmd = f"mpiexec -n {max_processes} DAMASK_grid -m {mat_file}.yaml -l {load_file}.yaml -g {geom_file}.vti"

In [ ]:
damask.util.run(cmd, wd=simulation_path)

### Run the first regridding (Python)

<div class="alert alert-warning">
Not done beyond this point.
</div>

In [ ]:
import regridding as rg
from shutil import copy
from DAMASK_to_DREAM3D import DAMASKtoDREAM3D

In [ ]:
os.chdir(simulation_folder)

In [ ]:
map_0to_rg, cells_rg, size_rg, increment_title = rg.regrid_geom(geom_file,load_file)
rg.write_geomRegridded(geom_file,increment_title,map_0to_rg,cells_rg,size_rg)
rg.write_h5OutRestart(geom_file,load_file,increment_title,map_0to_rg,cells_rg,isElastic=False)

Adding another load step of 0.1 compression strain

In [ ]:
os.chdir(simulation_folder)
rg.add_load_step(0.0076,3000,1000,1000)

 Preparing files for next deformation step in DAMASK

In [ ]:
copy(f'{geom_file}_regridded_{increment_title}.vti',f'{geom_file}.vti')
copy(f'{geom_file}_{load_file}_restart_regridded_{increment_title}.hdf5',f'{geom_file}_{load_file}_restart.hdf5')

### Run the second deformation step (DAMASK)

In [ ]:
restart_inc = increment_title.split('increment_')[1]
cmd = f'mpiexec -n {proc} DAMASK_grid -l {load_file}.yaml -g {geom_file}.vti -r {restart_inc}'

In [ ]:
damask.util.run(f'mpiexec -n {proc} DAMASK_grid -l {load_file}.yaml -g {geom_file}.vti -r {restart_inc}',wd=simulation_folder)

### Run the second regridding (Python)

In [ ]:
map_0to_rg, cells_rg, size_rg, increment_title = rg.regrid_geom(geom_file,load_file)
rg.write_geomRegridded(geom_file,increment_title,map_0to_rg,cells_rg,size_rg)
rg.write_h5OutRestart(geom_file,load_file,increment_title,map_0to_rg,cells_rg,isElastic=False)

copy(f'{geom_file}_regridded_{increment_title}.vti',f'{geom_file}.vti')
copy(f'{geom_file}_{load_file}_restart_regridded_{increment_title}.hdf5',f'{geom_file}_{load_file}_restart.hdf5')

restart_inc = increment_title.split('increment_')[1]

## Post-processing DAMASK data to SCORE

In [ ]:
os.chdir(simulation_folder)
d = damask.Result(f'{geom_file}_{load_file}.hdf5')
d.add_calculation('np.sum(#rho_mob#,axis=1) + np.sum(#rho_dip#,axis=1)','tot_density')          # calculating the total dislocation density

Adding the necessary post-processed datasets to the file.

In [ ]:
d = DAMASKtoDREAM3D(simulation_folder,f'{geom_file}_{load_file}.hdf5',geom_file,load_file)
d.DAMASKtoDREAM3D_deformed(int(restart_inc))
d.export_DREAM3D(int(restart_inc),'tot_density')

Remeshing the file for SCORE

Score needs regularly spaced grid and this function makes the dream3D geometry regularly spaced

In [ ]:
d.remesh_DREAM3D(int(restart_inc))

### Parameterize the nucleation model for SCORE.

In [ ]:
Parameterize the threshold dislocation density (here 3E15 m ** -2) above which nucleation sites are chosen.

In [ ]:
d.add_nucleation_max_rho(int(restart_inc), 3E15)

### Microstructure transfer between DAMASK and SCORE

In [ ]:
import json, h5py
import numpy as np

In [ ]:
pipeline_template_file = f"{cwd_path}/example_files/Create_CA_input_template.json"
pipeline_file = f"{cwd_path}/example_files/Create_CA_input.json"

geom_file_name = "Microstructure_for_paper"
load_file_name = "tensionX"
restart_inc = 6000

In [ ]:
with h5py.File(f"{simulation_path}/{geom_file_name}_{load_file_name}_remeshed_increment_{restart_inc}.dream3D") as h5r:
    tuple_dimensions = np.array(h5r['/DataContainers/SyntheticVolumeDataContainer/_SIMPL_GEOMETRY/DIMENSIONS']).tolist()
    voxel_spacing = np.array(h5r['/DataContainers/SyntheticVolumeDataContainer/_SIMPL_GEOMETRY/SPACING']).tolist()
tuple_dimensions, voxel_spacing

In [ ]:
with open(pipeline_template_file) as fp:
    data = json.load(fp)

    # create an instance from the template
    data['0']['InputFile'] = f"{simulation_path}/{geom_file_name}_{load_file_name}_remeshed_increment_{restart_inc}.dream3D"
    data['0']['InputFileDataContainerArrayProxy']['Data Containers'][0]['Attribute Matricies'][0]['Data Arrays'][0]['Tuple Dimensions'] = tuple_dimensions
    data['0']['InputFileDataContainerArrayProxy']['Data Containers'][0]['Attribute Matricies'][0]['Data Arrays'][1]['Tuple Dimensions'] = tuple_dimensions
    data['0']['InputFileDataContainerArrayProxy']['Data Containers'][0]['Attribute Matricies'][0]['Data Arrays'][2]['Tuple Dimensions'] = tuple_dimensions
    data['8']['OutputFile'] = f"{simulation_path}/{geom_file_name}_{load_file_name}_remeshed_increment_{restart_inc}_segmented.dream3D"

with open(pipeline_file, 'w') as fp:
    json.dump(data,fp,indent=4)

In [ ]:
"""
os.chdir(dream_3d_bin_path)                                                                                             
cmd = f'./PipelineRunner -p {dream_3d_pipeline_file}'                                                                   
p = subprocess.Popen(cmd,shell=True)                                                                                    
while p.poll() == None:
    p.poll()                                                                                                                                                                                                                                     
if p.poll() == 0:                                                                                                       
   print(f'Microstructure segmented. Run SCORE!!')
"""

## Recrystallization simulation (SCORE)

### Parameterize the recrystallization simulation (Python)

import os
import shutil

import numpy as np
from pint import Quantity
from score_config import ScoreConfig
from score_config_utils import ureg

In [ ]:
MY_MATERIAL: list[tuple[str, Quantity]] = [
    (
        "melting_temperature",
        ureg.Quantity(np.float64(650.0), ureg.degC).to(ureg.kelvin),
    ),
    (
        "shear_modulus_zero",
        ureg.Quantity(np.float64(26.0), ureg.gigapascal).to(ureg.pascal),
    ),
    ("nadal_lepoac_a", ureg.Quantity(np.float64(0.469))),
    ("nadal_lepoac_zeta", ureg.Quantity(np.float64(0.04))),
    ("burgers_vector_zero", ureg.Quantity(np.float64(0.286), ureg.nanometer)),
    # shown here how to consider a constant Burgers vector
    ("lattice_expansion_second", ureg.Quantity(0.0, (ureg.kelvin**-2))),
    ("lattice_expansion_first", ureg.Quantity(0.0, (ureg.kelvin**-1))),
    ("lattice_expansion_null", ureg.Quantity(0.0, ureg.nanometer)),
]

MY_USER: list[str] = [
    "Vitesh Shah",
    "Markus Kühbach",
    "Sebastian Samberger",
    "Georg Falkinger",
    "Franz Roters",
]

MY_INIT_DAMASK: dict[str, str] = {
    "file_path": f"{os.getcwd()}/../../examples/damask/Tphase_Vf_01_tensionX_remeshed_increment_19000_segmented.dream3D"
}

MY_MOBILITY_RH: list[tuple[str, Quantity]] = [
    (
        "m_null",
        ureg.Quantity(np.float64(300.0), ureg.meter**4 / (ureg.joule * ureg.second)),
    ),
    ("enthalpy", ureg.Quantity(np.float64(2.0314), ureg.eV)),
    # ("c1", ureg.Quantity(np.float64(0.9))),
    # ("c2", ureg.Quantity(np.float64(5.))),
    # ("c3", ureg.Quantity(np.float64(9.))),
]

MY_TIME_TEMPERATURE: list[tuple[str, Quantity]] = [
    (
        "time",
        ureg.Quantity(
            np.asarray(
                [
                    0.0,
                    5.8,
                    100.8,
                    103.7,
                    105.0,
                    200.8,
                    205.8,
                    208.7,
                    300.0,
                    305.8,
                    405.8,
                    408.7,
                    600.0,
                ],
                np.float64,
            ),
            ureg.second,
        ),
    ),
    (
        "temperature",
        ureg.Quantity(
            np.asarray(
                [
                    20.0,
                    500.0,
                    500.0,
                    20.0,
                    20.0,
                    500.0,
                    500.0,
                    20.0,
                    20.0,
                    500.0,
                    500.0,
                    20.0,
                    20.0,
                ],
                np.float64,
            ),
            ureg.degC,
        ).to(ureg.kelvin),
    ),
]

MY_GRID_CUBOIDAL: list[tuple[str, Quantity]] = [
    ("extent", ureg.Quantity(np.asarray([19, 15, 11], np.uint32))),
    ("cell_dimensions", ureg.Quantity(np.float64(0.5), ureg.micrometer)),
]

MY_GRID_DAMASK: list[tuple[str, Quantity]] = [
    ("extent", ureg.Quantity(np.asarray([128, 52, 64], np.uint32))),
    ("cell_dimensions", ureg.Quantity(np.float64(0.4982421), ureg.micrometer)),
]

In [ ]:
print("Generating a NeXus/HDF5 configuration file for examples/amag")
nxs = ScoreConfig(1)
nxs.user(MY_USER)
nxs.sample("Al")
nxs.material(MY_MATERIAL)
nxs.deformation("damask", MY_INIT_DAMASK)
nxs.nucleation("damask")
nxs.mobility("rollett_holm", MY_MOBILITY_RH)
nxs.time_temperature(MY_TIME_TEMPERATURE)
nxs.discretization(MY_GRID_DAMASK)
nxs.numerics()
nxs.get_config()
nxs.write_config_hfive_file()

### Run the recrystallization simulation (SCORE)

In [ ]:
# currently pointing to the wrong inputs, just for testing
simid = 1
score_config_file = f"{cwd_path}/../damask/SCORE.Config.SimID.{simid}.nxs"
score_input_file = f"{cwd_path}/../Tphase_Vf_01_tensionX_remeshed_increment_19000_segmented.dream3d"

In [ ]:
cmd = f"mpiexec -n 1 ./../../build/score {simid} {score_config_file}"

In [ ]:
p = subprocess.Popen(cmd, shell=True)

## Inspect the simulation results (e.g. H5Web)

In [ ]:
# from jupyterlab_h5web import H5Web
# H5Web(f"{cwd_path}/SCORE.Results.SimID.{simid}.nxs")

***
This completes the tutorial.